In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import os

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [48]:
# Step 1: Load the dataset
datapath = os.path.join("/content/drive/MyDrive/Colab Notebooks/Machine Learning/project test/")

''' Dataset choose from:
original_fraud_data
essentialCleaning_fraud_data
afterPreprocessed_fraud_data
creditcard
'''
dataset_name = 'afterPreprocessed_fraud_data'

fraudTrain = pd.read_csv(datapath + dataset_name + '.csv')

In [49]:
if dataset_name in ['original_fraud_data', 'essentialCleaning_fraud_data', 'afterPreprocessed_fraud_data']:
  # Step 2: data preprocessing
  '''
  Done by creating different dataset csv file to evaluate
  3 csv files:
  1. original_fraud_data.csv : Original data without any cleaning and preprocessing steps
  2. essentialCleaning_fraud_data.csv : Done with only essential cleaning steps
  3. afterPreprocessed_fraud_data.csv : Done with feature engineering
  '''

  # Step 3: Define target
  y = fraudTrain['is_fraud']

  # Step 4: Define input features without target
  X = fraudTrain.drop(columns=['is_fraud'])

  # Step 5: Detect column types
  categorical_cols = X.select_dtypes(include='object').columns.tolist()
  numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

  # Step 6: Preprocessor
  preprocessor = ColumnTransformer([
      ('cat_encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_cols)
  ], remainder='passthrough')

elif dataset_name == 'creditcard':
  # Step 2: data preprocessing
  '''
  No data preprocessing step as all values ​​are cleaned and floating point except the target
  '''
  # Step 3: Define target
  y = fraudTrain['Class']

  # Step 4: Define input features without target
  X = fraudTrain.drop(columns=['Class'])

  # Step 5: Detect column types
  numerical_cols = X.select_dtypes(include=['float64']).columns.tolist()

  # Step 6: Preprocessor
  ''' No categorical features are needed to preprocess'''
else:
  print("Invalid dataset name")

In [50]:
# Step 7: Pipeline
steps = []
if dataset_name in ['original_fraud_data', 'essentialCleaning_fraud_data', 'afterPreprocessed_fraud_data']:
    steps.append(('preprocessor', preprocessor))

steps.append(('imputer', SimpleImputer(strategy='median')))
steps.append(('classifier', XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=7,
    subsample=0.9,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss'
)))

model_pipeline = Pipeline(steps)

# Step 8: Split into train/test set and train
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
model_pipeline.fit(X_train, y_train)
y_pred = model_pipeline.predict(X_test)

# Step 9: Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.9987120921946617
Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00    386718
           1       0.96      0.82      0.88      2285

    accuracy                           1.00    389003
   macro avg       0.98      0.91      0.94    389003
weighted avg       1.00      1.00      1.00    389003

Confusion Matrix:
 [[386635     83]
 [   418   1867]]


In [52]:
# Step 10: Set a new threshold for sensitive FN cases
y_proba = model_pipeline.predict_proba(X_test)[:, 1]
threshold = 0.2
y_pred_newThreshold = (y_proba >= threshold).astype(int)
print(f"Threshold: {threshold}")
print("Accuracy:", accuracy_score(y_test, y_pred_newThreshold))
print("Classification Report:\n", classification_report(y_test, y_pred_newThreshold))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_newThreshold))

Threshold: 0.2
Accuracy: 0.9985192916250003
Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00    386718
           1       0.88      0.87      0.87      2285

    accuracy                           1.00    389003
   macro avg       0.94      0.93      0.94    389003
weighted avg       1.00      1.00      1.00    389003

Confusion Matrix:
 [[386443    275]
 [   301   1984]]
